## Parameters



In [87]:
%%writefile parameters.py
"""
Parameters for stable FDM simulation.
"""

import numpy as np

class Parameters:
    def __init__(self):
        self.gamma = 1.4
        self.R = 287.0

        # Smaller grid for testing
        self.Nx = 200  # Reduced from 500
        self.Ny = 40   # Reduced from 100
        self.Lx = 10.0
        self.Ly = 1.0

        # Lower CFL for stability
        self.CFL = 0.2
        self.t_final = 0.5  # Shorter time for testing
        self.t_print = 0.05

        self.output_dir = "output"
        self.update_grid()

    def update_grid(self):
        self.dx = self.Lx / (self.Nx - 1)
        self.dy = self.Ly / (self.Ny - 1)

params = Parameters()

Overwriting parameters.py


## Mesh

In [88]:
%%writefile mesh.py
"""
Mesh and grid generation - FDM version with nodes.
"""

import numpy as np
from parameters import params

class Mesh:
    def __init__(self):
        self.Nx = params.Nx
        self.Ny = params.Ny
        self.Lx = params.Lx
        self.Ly = params.Ly
        self.dx = params.dx
        self.dy = params.dy

        # Grid points (nodes) for FDM
        self.x_nodes = np.linspace(0, self.Lx, self.Nx)
        self.y_nodes = np.linspace(0, self.Ly, self.Ny)
        self.X, self.Y = np.meshgrid(self.x_nodes, self.y_nodes, indexing='ij')

        print(f"FDM Grid: {self.Nx} x {self.Ny} nodes")
        print(f"Domain: x∈[0,{self.Lx}], y∈[0,{self.Ly}]")
        print(f"dx={self.dx:.6f}, dy={self.dy:.6f}")

# Global mesh object
mesh = Mesh()

Overwriting mesh.py


## state

In [89]:
%%writefile state.py
"""
Compressible flow state variables at grid nodes.
"""

import numpy as np
from parameters import params
import mesh

class FlowState:
    __slots__ = ['gamma', 'Nx', 'Ny', 'Q', 'rho', 'u', 'v', 'p']

    def __init__(self):
        self.gamma = params.gamma
        self.Nx = mesh.mesh.Nx
        self.Ny = mesh.mesh.Ny

        # Conservative variables Q = [rho, rho*u, rho*v, E] at nodes
        self.Q = np.zeros((4, self.Nx, self.Ny))

        # Primitive variables at nodes
        self.rho = np.ones((self.Nx, self.Ny))
        self.u = np.zeros((self.Nx, self.Ny))
        self.v = np.zeros((self.Nx, self.Ny))
        self.p = np.ones((self.Nx, self.Ny))

    def prim_to_conservative(self):
        """Convert primitive to conservative"""
        self.Q[0] = self.rho
        self.Q[1] = self.rho * self.u
        self.Q[2] = self.rho * self.v
        kinetic = 0.5 * self.rho * (self.u**2 + self.v**2)
        self.Q[3] = self.p / (self.gamma - 1.0) + kinetic

    def conservative_to_primitive(self):
        """Convert conservative to primitive"""
        np.maximum(self.Q[0], 1e-10, out=self.rho)
        self.u = self.Q[1] / self.rho
        self.v = self.Q[2] / self.rho

        kinetic = 0.5 * self.rho * (self.u**2 + self.v**2)
        e_int = self.Q[3] / self.rho - kinetic
        np.maximum(e_int, 1e-10, out=e_int)
        self.p = (self.gamma - 1.0) * self.rho * e_int
        np.maximum(self.p, 1e-10, out=self.p)

    def get_mach_number(self):
        """Compute Mach number"""
        a = np.sqrt(self.gamma * self.p / (self.rho + 1e-14))
        speed = np.sqrt(self.u**2 + self.v**2)
        return speed / (a + 1e-14)

# Global state
state = FlowState()

Overwriting state.py


## Flux

In [90]:
%%writefile flux.py
"""
Physical flux functions for Euler equations.
"""

import numpy as np
from parameters import params

def flux_x(Q, gamma=params.gamma):
    """Flux in x-direction"""
    rho = np.maximum(Q[0], 1e-10)
    u = Q[1] / rho
    v = Q[2] / rho

    kinetic = 0.5 * rho * (u**2 + v**2)
    e_int = Q[3] / rho - kinetic
    e_int = np.maximum(e_int, 1e-10)
    p = (gamma - 1.0) * rho * e_int
    p = np.maximum(p, 1e-10)

    F = np.zeros_like(Q)
    F[0] = Q[1]
    F[1] = Q[1] * u + p
    F[2] = Q[1] * v
    F[3] = u * (Q[3] + p)
    return F

def flux_y(Q, gamma=params.gamma):
    """Flux in y-direction"""
    rho = np.maximum(Q[0], 1e-10)
    u = Q[1] / rho
    v = Q[2] / rho

    kinetic = 0.5 * rho * (u**2 + v**2)
    e_int = Q[3] / rho - kinetic
    e_int = np.maximum(e_int, 1e-10)
    p = (gamma - 1.0) * rho * e_int
    p = np.maximum(p, 1e-10)

    G = np.zeros_like(Q)
    G[0] = Q[2]
    G[1] = Q[2] * u
    G[2] = Q[2] * v + p
    G[3] = v * (Q[3] + p)
    return G

Overwriting flux.py


## WENO reconstruction

In [105]:
%%writefile weno5_fdm_fixed.py
"""
Simplified WENO5 scheme - stable version.
"""

import numpy as np
import mesh

def weno5_flux_x(Q, gamma=1.4):
    from flux import flux_x

    Nx, Ny = Q.shape[1], Q.shape[2]
    eps = 1e-6

    F = flux_x(Q, gamma)

    # ----------- LAX-FRIEDRICHS FLUX SPLITTING -----------
    # Compute max wave speed (important!)
    rho = Q[0]
    u = Q[1] / rho
    E = Q[3]
    p = (gamma - 1) * (E - 0.5 * rho * u**2)

    a = np.sqrt(gamma * p / rho)
    alpha = np.max(np.abs(u) + a)

    F_plus  = 0.5 * (F + alpha * Q)
    F_minus = 0.5 * (F - alpha * Q)

    # ----------- ADD GHOST CELLS -----------
    def add_ghost(F):
        Fg = np.zeros((4, Nx+6, Ny))
        Fg[:, 3:3+Nx, :] = F

        for i in range(3):
            Fg[:, 2-i, :] = F[:, 0, :]
            Fg[:, 3+Nx+i, :] = F[:, -1, :]

        return Fg

    Fp = add_ghost(F_plus)
    Fm = add_ghost(F_minus)

    dFdx = np.zeros_like(F)

    # Linear weights
    d0, d1, d2 = 0.1, 0.6, 0.3

    for n in range(4):
        for i in range(Nx):
            idx = i + 3

            # ========================
            # LEFT FLUX (F+)
            # ========================
            f = Fp[n]

            f0, f1, f2 = f[idx-3], f[idx-2], f[idx-1]
            f3, f4, f5 = f[idx],   f[idx+1], f[idx+2]

            q0 = (1/3)*f0 - (7/6)*f1 + (11/6)*f2
            q1 = -(1/6)*f1 + (5/6)*f2 + (1/3)*f3
            q2 = (1/3)*f2 + (5/6)*f3 - (1/6)*f4

            beta0 = (13/12)*(f0 - 2*f1 + f2)**2 + 0.25*(f0 - 4*f1 + 3*f2)**2
            beta1 = (13/12)*(f1 - 2*f2 + f3)**2 + 0.25*(f1 - f3)**2
            beta2 = (13/12)*(f2 - 2*f3 + f4)**2 + 0.25*(3*f2 - 4*f3 + f4)**2

            a0 = d0 / (beta0 + eps)**2
            a1 = d1 / (beta1 + eps)**2
            a2 = d2 / (beta2 + eps)**2

            w0 = a0 / (a0 + a1 + a2)
            w1 = a1 / (a0 + a1 + a2)
            w2 = a2 / (a0 + a1 + a2)

            f_left = w0*q0 + w1*q1 + w2*q2

            # ========================
            # RIGHT FLUX (F-)
            # ========================
            f = Fm[n]

            f0, f1, f2 = f[idx+2], f[idx+1], f[idx]
            f3, f4, f5 = f[idx-1], f[idx-2], f[idx-3]

            q0 = (1/3)*f0 - (7/6)*f1 + (11/6)*f2
            q1 = -(1/6)*f1 + (5/6)*f2 + (1/3)*f3
            q2 = (1/3)*f2 + (5/6)*f3 - (1/6)*f4

            beta0 = (13/12)*(f0 - 2*f1 + f2)**2 + 0.25*(f0 - 4*f1 + 3*f2)**2
            beta1 = (13/12)*(f1 - 2*f2 + f3)**2 + 0.25*(f1 - f3)**2
            beta2 = (13/12)*(f2 - 2*f3 + f4)**2 + 0.25*(3*f2 - 4*f3 + f4)**2

            a0 = d0 / (beta0 + eps)**2
            a1 = d1 / (beta1 + eps)**2
            a2 = d2 / (beta2 + eps)**2

            w0 = a0 / (a0 + a1 + a2)
            w1 = a1 / (a0 + a1 + a2)
            w2 = a2 / (a0 + a1 + a2)

            f_right = w0*q0 + w1*q1 + w2*q2

            dFdx[n, i, :] = (f_left + f_right) / mesh.mesh.dx

    return dFdx

Overwriting weno5_fdm_fixed.py


## Compact FDM

In [92]:
%%writefile compact_fdm.py
"""
Compact Difference Scheme for FDM - 6th order.
Fixed version with proper array sizing.
"""

import numpy as np
import mesh

def compact_derivative_x(F, dx):
    """
    6th-order compact scheme for first derivative.
    Solves tridiagonal system.
    """
    Nx = F.shape[1]
    alpha = 1.0/3.0  # Compact scheme parameter

    # RHS for compact scheme (6th order)
    a = 14.0/9.0
    b = 1.0/9.0

    # Initialize RHS
    rhs = np.zeros_like(F)

    # Interior points
    for i in range(2, Nx-2):
        rhs[:, i, :] = a * (F[:, i+1, :] - F[:, i-1, :]) / (2*dx) + \
                       b * (F[:, i+2, :] - F[:, i-2, :]) / (4*dx)

    # Boundaries - use explicit 4th order
    rhs[:, 0, :] = ( -25*F[:, 0, :] + 48*F[:, 1, :] - 36*F[:, 2, :] + 16*F[:, 3, :] - 3*F[:, 4, :] ) / (12*dx)
    rhs[:, 1, :] = ( -3*F[:, 0, :] - 10*F[:, 1, :] + 18*F[:, 2, :] - 6*F[:, 3, :] + F[:, 4, :] ) / (12*dx)
    rhs[:, Nx-2, :] = ( -F[:, Nx-5, :] + 6*F[:, Nx-4, :] - 18*F[:, Nx-3, :] + 10*F[:, Nx-2, :] + 3*F[:, Nx-1, :] ) / (12*dx)
    rhs[:, Nx-1, :] = ( 3*F[:, Nx-5, :] - 16*F[:, Nx-4, :] + 36*F[:, Nx-3, :] - 48*F[:, Nx-2, :] + 25*F[:, Nx-1, :] ) / (12*dx)

    # Solve tridiagonal system for each variable and each y-slice
    dFdx = np.zeros_like(F)

    for n in range(4):  # For each conservative variable
        for j in range(F.shape[2]):  # For each y-slice
            # Solve tridiagonal system using Thomas algorithm
            # Size Nx
            a_diag = alpha * np.ones(Nx-1)  # Sub-diagonal (size Nx-1)
            b_diag = np.ones(Nx)            # Main diagonal (size Nx)
            c_diag = alpha * np.ones(Nx-1)  # Super-diagonal (size Nx-1)

            # Forward sweep
            c_prime = np.zeros(Nx-1)
            rhs_prime = np.zeros(Nx)

            # First row
            rhs_prime[0] = rhs[n, 0, j] / b_diag[0]

            # Forward elimination
            for i in range(1, Nx):
                if i < Nx-1:
                    c_prime[i] = c_diag[i-1] / b_diag[i-1]
                b_diag[i] = b_diag[i] - a_diag[i-1] * c_prime[i-1]
                rhs_prime[i] = (rhs[n, i, j] - a_diag[i-1] * rhs_prime[i-1]) / b_diag[i]

            # Back substitution
            dFdx[n, Nx-1, j] = rhs_prime[Nx-1]
            for i in range(Nx-2, -1, -1):
                dFdx[n, i, j] = rhs_prime[i] - c_prime[i] * dFdx[n, i+1, j]

    return dFdx


def compact_derivative_y(G, dy):
    """
    6th-order compact scheme for first derivative in y-direction.
    """
    Ny = G.shape[2]
    alpha = 1.0/3.0

    a = 14.0/9.0
    b = 1.0/9.0

    rhs = np.zeros_like(G)

    # Interior points
    for j in range(2, Ny-2):
        rhs[:, :, j] = a * (G[:, :, j+1] - G[:, :, j-1]) / (2*dy) + \
                       b * (G[:, :, j+2] - G[:, :, j-2]) / (4*dy)

    # Boundaries
    rhs[:, :, 0] = ( -25*G[:, :, 0] + 48*G[:, :, 1] - 36*G[:, :, 2] + 16*G[:, :, 3] - 3*G[:, :, 4] ) / (12*dy)
    rhs[:, :, 1] = ( -3*G[:, :, 0] - 10*G[:, :, 1] + 18*G[:, :, 2] - 6*G[:, :, 3] + G[:, :, 4] ) / (12*dy)
    rhs[:, :, Ny-2] = ( -G[:, :, Ny-5] + 6*G[:, :, Ny-4] - 18*G[:, :, Ny-3] + 10*G[:, :, Ny-2] + 3*G[:, :, Ny-1] ) / (12*dy)
    rhs[:, :, Ny-1] = ( 3*G[:, :, Ny-5] - 16*G[:, :, Ny-4] + 36*G[:, :, Ny-3] - 48*G[:, :, Ny-2] + 25*G[:, :, Ny-1] ) / (12*dy)

    # Solve tridiagonal system for each variable and each x-slice
    dGdy = np.zeros_like(G)

    for n in range(4):
        for i in range(G.shape[1]):
            # Size Ny
            a_diag = alpha * np.ones(Ny-1)
            b_diag = np.ones(Ny)
            c_diag = alpha * np.ones(Ny-1)

            # Forward sweep
            c_prime = np.zeros(Ny-1)
            rhs_prime = np.zeros(Ny)

            rhs_prime[0] = rhs[n, i, 0] / b_diag[0]

            for j in range(1, Ny):
                if j < Ny-1:
                    c_prime[j] = c_diag[j-1] / b_diag[j-1]
                b_diag[j] = b_diag[j] - a_diag[j-1] * c_prime[j-1]
                rhs_prime[j] = (rhs[n, i, j] - a_diag[j-1] * rhs_prime[j-1]) / b_diag[j]

            # Back substitution
            dGdy[n, i, Ny-1] = rhs_prime[Ny-1]
            for j in range(Ny-2, -1, -1):
                dGdy[n, i, j] = rhs_prime[j] - c_prime[j] * dGdy[n, i, j+1]

    return dGdy


def compute_rhs_compact(Q, dx, dy, gamma=1.4):
    """
    Compute RHS using 6th-order compact scheme.
    """
    from flux import flux_x, flux_y

    # Compute fluxes
    F = flux_x(Q, gamma)
    G = flux_y(Q, gamma)

    # Add ghost points for boundaries
    Nx, Ny = Q.shape[1], Q.shape[2]

    # Extend F in x-direction (need 2 ghost points on each side for compact scheme)
    F_ext = np.zeros((4, Nx+4, Ny))
    F_ext[:, 2:2+Nx, :] = F

    # Fill ghosts (transmissive)
    F_ext[:, 0, :] = F[:, 0, :]
    F_ext[:, 1, :] = F[:, 0, :]
    F_ext[:, -2, :] = F[:, -1, :]
    F_ext[:, -1, :] = F[:, -1, :]

    # Extend G in y-direction
    G_ext = np.zeros((4, Nx, Ny+4))
    G_ext[:, :, 2:2+Ny] = G

    G_ext[:, :, 0] = G[:, :, 0]
    G_ext[:, :, 1] = G[:, :, 0]
    G_ext[:, :, -2] = G[:, :, -1]
    G_ext[:, :, -1] = G[:, :, -1]

    # Compute derivatives using compact scheme
    dFdx = compact_derivative_x(F_ext, dx)
    dGdy = compact_derivative_y(G_ext, dy)

    # Trim ghost points (remove 2 from each side)
    dFdx = dFdx[:, 2:2+Nx, :]
    dGdy = dGdy[:, :, 2:2+Ny]

    return -(dFdx + dGdy)

Overwriting compact_fdm.py


## MUSCL with FDM

In [93]:
%%writefile muscl_fdm.py
"""
MUSCL scheme for FDM - much faster than WENO5.
"""

import numpy as np
import mesh

def minmod(a, b):
    """Minmod limiter"""
    return 0.5 * (np.sign(a) + np.sign(b)) * np.minimum(np.abs(a), np.abs(b))

def muscl_flux_x(Q, gamma=1.4):
    """
    MUSCL reconstruction for flux in x-direction.
    Much faster than WENO5.
    """
    from flux import flux_x

    Nx, Ny = Q.shape[1], Q.shape[2]

    # Compute fluxes at all points
    F = flux_x(Q, gamma)

    # Add ghost points (2 on each side for MUSCL)
    Fg = np.zeros((4, Nx+4, Ny))
    Fg[:, 2:2+Nx, :] = F

    # Fill ghosts (transmissive)
    Fg[:, 0:2, :] = F[:, 0:1, :]
    Fg[:, -2:, :] = F[:, -1:, :]

    # Initialize flux at interfaces
    F_left = np.zeros((4, Nx+1, Ny))
    F_right = np.zeros((4, Nx+1, Ny))

    for n in range(4):
        for i in range(Nx+1):
            idx = i + 1  # Index in ghost array

            # Get stencil values
            f0 = Fg[n, idx-1, :]
            f1 = Fg[n, idx, :]
            f2 = Fg[n, idx+1, :]

            # Simple MUSCL reconstruction with minmod
            slope_left = f1 - f0
            slope_right = f2 - f1
            slope = minmod(slope_left, slope_right)

            # Left and right fluxes at interface
            F_left[n, i, :] = f1 - 0.5 * slope
            F_right[n, i, :] = f1 + 0.5 * slope

    # Compute flux derivative
    dFdx = np.zeros_like(Q)

    # Interior points
    for i in range(1, Nx-1):
        dFdx[:, i, :] = (F_right[:, i, :] - F_left[:, i, :]) / mesh.mesh.dx

    # Boundaries
    dFdx[:, 0, :] = (F[:, 1, :] - F[:, 0, :]) / mesh.mesh.dx
    dFdx[:, -1, :] = (F[:, -1, :] - F[:, -2, :]) / mesh.mesh.dx

    return dFdx


def muscl_flux_y(Q, gamma=1.4):
    """
    MUSCL reconstruction for flux in y-direction.
    """
    from flux import flux_y

    Nx, Ny = Q.shape[1], Q.shape[2]

    G = flux_y(Q, gamma)

    # Add ghost points
    Gg = np.zeros((4, Nx, Ny+4))
    Gg[:, :, 2:2+Ny] = G

    # Fill ghosts
    Gg[:, :, 0:2] = G[:, :, 0:1]
    Gg[:, :, -2:] = G[:, :, -1:]

    G_bottom = np.zeros((4, Nx, Ny+1))
    G_top = np.zeros((4, Nx, Ny+1))

    for n in range(4):
        for j in range(Ny+1):
            idx = j + 1

            g0 = Gg[n, :, idx-1]
            g1 = Gg[n, :, idx]
            g2 = Gg[n, :, idx+1]

            slope_left = g1 - g0
            slope_right = g2 - g1
            slope = minmod(slope_left, slope_right)

            G_bottom[n, :, j] = g1 - 0.5 * slope
            G_top[n, :, j] = g1 + 0.5 * slope

    # Compute y-derivative
    dGdy = np.zeros_like(Q)

    for j in range(1, Ny-1):
        dGdy[:, :, j] = (G_top[:, :, j] - G_bottom[:, :, j]) / mesh.mesh.dy

    dGdy[:, :, 0] = (G[:, :, 1] - G[:, :, 0]) / mesh.mesh.dy
    dGdy[:, :, -1] = (G[:, :, -1] - G[:, :, -2]) / mesh.mesh.dy

    return dGdy

Overwriting muscl_fdm.py


## simple central

In [94]:
%%writefile fdm_viscosity.py
"""
Central scheme with artificial viscosity for shock capturing.
"""

import numpy as np
import mesh

def compute_rhs_viscosity(Q, dx, dy, gamma=1.4, visc_coeff=0.5):
    """
    4th-order central with artificial viscosity.
    """
    from flux import flux_x, flux_y

    Nx, Ny = Q.shape[1], Q.shape[2]

    # Compute fluxes
    F = flux_x(Q, gamma)
    G = flux_y(Q, gamma)

    # Add ghost points
    Fg = np.zeros((4, Nx+4, Ny))
    Gg = np.zeros((4, Nx, Ny+4))

    Fg[:, 2:2+Nx, :] = F
    Gg[:, :, 2:2+Ny] = G

    # Transmissive BCs
    Fg[:, 0, :] = F[:, 0, :]
    Fg[:, 1, :] = F[:, 0, :]
    Fg[:, -2, :] = F[:, -1, :]
    Fg[:, -1, :] = F[:, -1, :]

    Gg[:, :, 0] = G[:, :, 0]
    Gg[:, :, 1] = G[:, :, 0]
    Gg[:, :, -2] = G[:, :, -1]
    Gg[:, :, -1] = G[:, :, -1]

    # 4th-order central difference
    dFdx = np.zeros_like(F)
    dGdy = np.zeros_like(G)

    # Interior
    for i in range(2, Nx-2):
        dFdx[:, i, :] = (-Fg[:, i+2, :] + 8*Fg[:, i+1, :] - 8*Fg[:, i-1, :] + Fg[:, i-2, :]) / (12*dx)

    # Boundaries
    dFdx[:, 0, :] = (F[:, 1, :] - F[:, 0, :]) / dx
    dFdx[:, 1, :] = (F[:, 2, :] - F[:, 0, :]) / (2*dx)
    dFdx[:, -2, :] = (F[:, -1, :] - F[:, -3, :]) / (2*dx)
    dFdx[:, -1, :] = (F[:, -1, :] - F[:, -2, :]) / dx

    # Artificial viscosity (4th-order dissipation)
    visc_x = np.zeros_like(Q)

    # Compute pressure sensor for shock detection
    p = Q[3] - 0.5*(Q[1]**2 + Q[2]**2)/np.maximum(Q[0], 1e-10)
    p = np.maximum((gamma-1)*p, 1e-10)

    # Shock sensor (normalized pressure gradient)
    dpdx = np.zeros_like(p)
    dpdx[2:-2, :] = np.abs(p[3:-1, :] - 2*p[2:-2, :] + p[1:-3, :])
    sensor = dpdx / (np.maximum(p, 1e-10) + 1e-10)
    sensor = np.minimum(sensor * 10, 1.0)  # Scale between 0 and 1

    # Add viscosity where shocks are detected
    for i in range(2, Nx-2):
        visc = visc_coeff * sensor[i, :] * (Q[:, i+2, :] - 4*Q[:, i+1, :] + 6*Q[:, i, :] - 4*Q[:, i-1, :] + Q[:, i-2, :]) / dx**2
        visc_x[:, i, :] = visc

    return -(dFdx + dGdy) + visc_x

Writing fdm_viscosity.py


## 4th- order Central FDM

In [95]:
%%writefile fdm_central.py
"""
4th-order central FDM scheme.
"""

import numpy as np
from flux import flux_x, flux_y

def compute_rhs_central(Q, dx, dy, gamma=1.4):
    """
    4th-order central difference for flux derivatives.
    """
    Nx, Ny = Q.shape[1], Q.shape[2]

    # Compute fluxes at all points
    F = flux_x(Q, gamma)
    G = flux_y(Q, gamma)

    # Add ghost points for boundary conditions
    Fg = np.zeros((4, Nx+4, Ny))
    Gg = np.zeros((4, Nx, Ny+4))

    Fg[:, 2:2+Nx, :] = F
    Gg[:, :, 2:2+Ny] = G

    # Transmissive BCs for ghosts
    for i in range(2):
        Fg[:, i, :] = F[:, 0, :]
        Fg[:, -1-i, :] = F[:, -1, :]
        Gg[:, :, i] = G[:, :, 0]
        Gg[:, :, -1-i] = G[:, :, -1]

    # 4th-order central difference
    dFdx = np.zeros_like(F)
    dGdy = np.zeros_like(G)

    # Interior points
    for i in range(2, Nx-2):
        dFdx[:, i, :] = (-Fg[:, i+2, :] + 8*Fg[:, i+1, :] - 8*Fg[:, i-1, :] + Fg[:, i-2, :]) / (12*dx)

    # Boundaries - 2nd order
    dFdx[:, 0, :] = (F[:, 1, :] - F[:, 0, :]) / dx
    dFdx[:, 1, :] = (F[:, 2, :] - F[:, 0, :]) / (2*dx)
    dFdx[:, Nx-2, :] = (F[:, Nx-1, :] - F[:, Nx-3, :]) / (2*dx)
    dFdx[:, Nx-1, :] = (F[:, Nx-1, :] - F[:, Nx-2, :]) / dx

    # Y-direction
    for j in range(2, Ny-2):
        dGdy[:, :, j] = (-Gg[:, :, j+2] + 8*Gg[:, :, j+1] - 8*Gg[:, :, j-1] + Gg[:, :, j-2]) / (12*dy)

    dGdy[:, :, 0] = (G[:, :, 1] - G[:, :, 0]) / dy
    dGdy[:, :, 1] = (G[:, :, 2] - G[:, :, 0]) / (2*dy)
    dGdy[:, :, Ny-2] = (G[:, :, Ny-1] - G[:, :, Ny-3]) / (2*dy)
    dGdy[:, :, Ny-1] = (G[:, :, Ny-1] - G[:, :, Ny-2]) / dy

    return -(dFdx + dGdy)

Overwriting fdm_central.py


## FLux-divergence

In [106]:
%%writefile flux_divergence.py
"""
Flux divergence for FDM - Choose scheme here.
"""

import numpy as np
from parameters import params
from weno5_fdm import weno5_flux_x, weno5_flux_y
from fdm_central import compute_rhs_central
from muscl_fdm import muscl_flux_x, muscl_flux_y
from compact_fdm import compute_rhs_compact

# Choose scheme: "WENO5" or "CENTRAL"
SCHEME = "WENO5"  # Change to "CENTRAL" for 4th-order central

def compute_rhs(Q, dx, dy):
    """Compute RHS using selected FDM scheme"""
    if SCHEME == "WENO5":
        # WENO5 reconstruction for fluxes
        dFdx = weno5_flux_x(Q, params.gamma)
        dGdy = weno5_flux_y(Q, params.gamma)
        return -(dFdx + dGdy)
    elif SCHEME == "MUSCL":
      # MUSCL - much faster
        dFdx = muscl_flux_x(Q, params.gamma)
        dGdy = muscl_flux_y(Q, params.gamma)
        return -(dFdx + dGdy)
    elif SCHEME == "CENTRAL":
        from fdm_central import compute_rhs_central
        return compute_rhs_central(Q, dx, dy, params.gamma)
    elif SCHEME == "VISCOSITY":
      from fdm_viscosity import compute_rhs_viscosity
      return compute_rhs_viscosity(Q, dx, dy, params.gamma, visc_coeff=0.5)
    else:  # COMPACT
        return compute_rhs_compact(Q, dx, dy, params.gamma)

Overwriting flux_divergence.py


## Boundary Conditions

In [97]:
%%writefile boundary.py
"""
Boundary conditions for FDM.
"""

import numpy as np

def apply_bc_x(Q):
    """Transmissive BCs in x for FDM"""
    # Left boundary
    Q[:, 0, :] = Q[:, 1, :]
    Q[:, 1, :] = Q[:, 2, :]  # For 4th-order stencil

    # Right boundary
    Q[:, -1, :] = Q[:, -2, :]
    Q[:, -2, :] = Q[:, -3, :]
    return Q

def apply_bc_y(Q):
    """Transmissive/Periodic BCs in y"""
    # Bottom boundary
    Q[:, :, 0] = Q[:, :, 1]
    Q[:, :, 1] = Q[:, :, 2]

    # Top boundary
    Q[:, :, -1] = Q[:, :, -2]
    Q[:, :, -2] = Q[:, :, -3]
    return Q

Overwriting boundary.py


## Time- stepping

In [98]:
%%writefile timestep.py
"""
CFL condition for FDM.
"""

import numpy as np
from parameters import params

def compute_dt(Q, dx, dy):
    """Compute stable time step for FDM"""
    rho = np.maximum(Q[0], 1e-10)
    u = Q[1] / rho
    v = Q[2] / rho

    kinetic = 0.5 * (u**2 + v**2)
    e_int = Q[3] / rho - kinetic
    e_int = np.maximum(e_int, 1e-10)
    p = (params.gamma - 1.0) * rho * e_int
    p = np.maximum(p, 1e-10)
    a = np.sqrt(params.gamma * p / (rho + 1e-14))

    max_speed_x = np.max(np.abs(u) + a)
    max_speed_y = np.max(np.abs(v) + a)
    max_speed = max(max_speed_x, max_speed_y)
    max_speed = max(max_speed, 1e-10)

    # Compact schemes need slightly lower CFL
    dt = 0.8 * params.CFL * min(dx, dy) / max_speed

    # Add a minimum dt to prevent complete stall
    dt = max(dt, 1e-8)

    return dt

Overwriting timestep.py


## Rk3- Time Integration

In [99]:
%%writefile rk3.py
"""
TVD Runge-Kutta 3 time integration for FDM.
"""

import numpy as np
from flux_divergence import compute_rhs
from boundary import apply_bc_x, apply_bc_y

def apply_boundary_conditions(Q):
    """Apply all boundary conditions"""
    Q = apply_bc_x(Q)
    Q = apply_bc_y(Q)
    return Q

def rk3_step(Q, dt, dx, dy):
    """Take one RK3 time step"""
    # Stage 1
    L0 = compute_rhs(Q, dx, dy)
    Q1 = Q + dt * L0
    Q1 = apply_boundary_conditions(Q1)
    Q1 = np.nan_to_num(Q1, nan=1e-10)
    Q1[0] = np.maximum(Q1[0], 1e-10)
    Q1[3] = np.maximum(Q1[3], 1e-10)

    # Stage 2
    L1 = compute_rhs(Q1, dx, dy)
    Q2 = 0.75 * Q + 0.25 * (Q1 + dt * L1)
    Q2 = apply_boundary_conditions(Q2)
    Q2 = np.nan_to_num(Q2, nan=1e-10)
    Q2[0] = np.maximum(Q2[0], 1e-10)
    Q2[3] = np.maximum(Q2[3], 1e-10)

    # Stage 3
    L2 = compute_rhs(Q2, dx, dy)
    Q3 = (1.0/3.0) * Q + (2.0/3.0) * (Q2 + dt * L2)
    Q3 = apply_boundary_conditions(Q3)
    Q3 = np.nan_to_num(Q3, nan=1e-10)
    Q3[0] = np.maximum(Q3[0], 1e-10)
    Q3[3] = np.maximum(Q3[3], 1e-10)

    return Q3

Overwriting rk3.py


## Intial conditions

In [100]:
%%writefile initial_conditions.py
"""
Initial conditions for Sod shock tube - FDM version.
"""

import numpy as np
import mesh
import state

def init_sod():
    """Initialize Sod shock tube on nodes"""
    X = mesh.mesh.X
    x_diaphragm = mesh.mesh.Lx / 2.0

    state.state.rho = np.where(X < x_diaphragm, 1.0, 0.125)
    state.state.u = np.zeros_like(X)
    state.state.v = np.zeros_like(X)
    state.state.p = np.where(X < x_diaphragm, 1.0, 0.1)

    state.state.prim_to_conservative()

    print("Sod shock tube initialized (FDM)")
    print(f"  Diaphragm at x = {x_diaphragm:.2f}")
    print(f"  Left:  rho=1.0, p=1.0")
    print(f"  Right: rho=0.125, p=0.1")

Overwriting initial_conditions.py


## Exact Sod

In [101]:
%%writefile exact_sod.py
"""
Exact solution for Sod shock tube.
"""

import numpy as np
from parameters import params

def exact_sod_solution(x, t, gamma=params.gamma, x_diaphragm=0.5):
    """Exact solution for Sod shock tube"""
    rhoL, uL, pL = 1.0, 0.0, 1.0
    rhoR, uR, pR = 0.125, 0.0, 0.1

    aL = np.sqrt(gamma * pL / rhoL)
    aR = np.sqrt(gamma * pR / rhoR)

    def fL(p):
        if p > pL:
            A = 2.0 / ((gamma + 1.0) * rhoL)
            B = (gamma - 1.0) / (gamma + 1.0) * pL
            return (p - pL) * np.sqrt(A / (p + B + 1e-14))
        else:
            return (2.0 * aL / (gamma - 1.0)) * ((p / pL)**((gamma - 1.0) / (2.0 * gamma)) - 1.0)

    def fR(p):
        if p > pR:
            A = 2.0 / ((gamma + 1.0) * rhoR)
            B = (gamma - 1.0) / (gamma + 1.0) * pR
            return (p - pR) * np.sqrt(A / (p + B + 1e-14))
        else:
            return (2.0 * aR / (gamma - 1.0)) * ((p / pR)**((gamma - 1.0) / (2.0 * gamma)) - 1.0)

    def f(p):
        return fL(p) + fR(p) + (uR - uL)

    def df(p):
        dp = max(1e-6 * p, 1e-8)
        return (f(p + dp) - f(p - dp)) / (2.0 * dp)

    p_star = 0.5 * (pL + pR)
    for _ in range(50):
        dp = -f(p_star) / (df(p_star) + 1e-14)
        p_star += dp
        if abs(dp) < 1e-12:
            break

    u_star = 0.5 * (uL + uR) + 0.5 * (fR(p_star) - fL(p_star))

    aL_star = aL * (p_star / pL)**((gamma - 1.0) / (2.0 * gamma))
    S_HL = uL - aL
    S_TL = u_star - aL_star
    S_contact = u_star
    S_R = uR + aR * np.sqrt((gamma + 1.0) / (2.0 * gamma) * (p_star / pR) +
                            (gamma - 1.0) / (2.0 * gamma))

    rhoL_star = rhoL * (p_star / pL)**(1.0 / gamma)
    rhoR_star = rhoR * ((p_star / pR + (gamma - 1.0) / (gamma + 1.0)) /
                        ((gamma - 1.0) / (gamma + 1.0) * p_star / pR + 1.0))

    if t <= 0:
        return (np.where(x < x_diaphragm, rhoL, rhoR),
                np.where(x < x_diaphragm, uL, uR),
                np.where(x < x_diaphragm, pL, pR))

    xi = (x - x_diaphragm) / (t + 1e-14)
    rho = np.zeros_like(x)
    u = np.zeros_like(x)
    p = np.zeros_like(x)

    for i, s in enumerate(xi):
        if s <= S_HL:
            rho[i], u[i], p[i] = rhoL, uL, pL
        elif s <= S_TL:
            u_tmp = 2.0 / (gamma + 1.0) * (aL + (gamma - 1.0) / 2.0 * uL + s)
            a_tmp = aL + (gamma - 1.0) / 2.0 * (uL - u_tmp)
            rho[i] = rhoL * (a_tmp / aL)**(2.0 / (gamma - 1.0))
            u[i] = u_tmp
            p[i] = pL * (a_tmp / aL)**(2.0 * gamma / (gamma - 1.0))
        elif s <= S_contact:
            rho[i], u[i], p[i] = rhoL_star, u_star, p_star
        elif s <= S_R:
            rho[i], u[i], p[i] = rhoR_star, u_star, p_star
        else:
            rho[i], u[i], p[i] = rhoR, uR, pR

    return rho, u, p

Overwriting exact_sod.py


## Plotting

In [102]:
%%writefile plotting.py
"""
Plotting functions for FDM.
"""

import numpy as np
import matplotlib.pyplot as plt
import os
from parameters import params
import mesh
import state
from exact_sod import exact_sod_solution

def ensure_output_dir():
    """Ensure output directory exists"""
    os.makedirs(f"{params.output_dir}/plots", exist_ok=True)

def plot_verification(t, step):
    """Plot numerical vs exact solution"""
    ensure_output_dir()

    rho = state.state.rho
    u = state.state.u
    p = state.state.p

    x = mesh.mesh.x_nodes  # Use nodes for FDM
    rho_num = rho.mean(axis=1)
    u_num = u.mean(axis=1)
    p_num = p.mean(axis=1)

    x_diaphragm = mesh.mesh.Lx / 2.0
    rho_ex, u_ex, p_ex = exact_sod_solution(x, t, x_diaphragm=x_diaphragm)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f"Sod Shock Tube (FDM) - t={t:.4f}, step={step}", fontsize=14)

    axes[0].plot(x, rho_ex, 'k-', lw=2, label='Exact')
    axes[0].plot(x, rho_num, 'ro--', ms=3, lw=1, alpha=0.7, label='FDM-WENO5')
    axes[0].axvline(x=x_diaphragm, color='gray', linestyle='--', alpha=0.5)
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('Density')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim([0, mesh.mesh.Lx])

    axes[1].plot(x, u_ex, 'k-', lw=2, label='Exact')
    axes[1].plot(x, u_num, 'go--', ms=3, lw=1, alpha=0.7, label='FDM-WENO5')
    axes[1].axvline(x=x_diaphragm, color='gray', linestyle='--', alpha=0.5)
    axes[1].set_xlabel('x')
    axes[1].set_ylabel('Velocity')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim([0, mesh.mesh.Lx])

    axes[2].plot(x, p_ex, 'k-', lw=2, label='Exact')
    axes[2].plot(x, p_num, 'bo--', ms=3, lw=1, alpha=0.7, label='FDM-WENO5')
    axes[2].axvline(x=x_diaphragm, color='gray', linestyle='--', alpha=0.5)
    axes[2].set_xlabel('x')
    axes[2].set_ylabel('Pressure')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    axes[2].set_xlim([0, mesh.mesh.Lx])

    plt.tight_layout()
    filename = f"{params.output_dir}/plots/verification_t{t:.4f}.png"
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Verification plot saved: {filename}")

def plot_schlieren(t, step, k=10.0):
    """Plot numerical schlieren"""
    ensure_output_dir()

    rho = state.state.rho
    drho_dx = np.gradient(rho, mesh.mesh.dx, axis=0)
    drho_dy = np.gradient(rho, mesh.mesh.dy, axis=1)
    grad_mag = np.sqrt(drho_dx**2 + drho_dy**2)
    grad_norm = grad_mag / (grad_mag.max() + 1e-14)
    schlieren = np.exp(-k * grad_norm)

    fig, ax = plt.subplots(figsize=(12, 3))
    im = ax.imshow(schlieren.T, origin='lower',
                   extent=[0, mesh.mesh.Lx, 0, mesh.mesh.Ly],
                   cmap='gray', aspect='auto')
    plt.colorbar(im, ax=ax)
    ax.axvline(x=mesh.mesh.Lx/2, color='red', linestyle='--', alpha=0.5)
    ax.set_title(f"Schlieren |∇ρ| (FDM) - t={t:.4f}")
    ax.set_xlabel('x')
    ax.set_ylabel('y')

    filename = f"{params.output_dir}/plots/schlieren_t{t:.4f}.png"
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Schlieren plot saved: {filename}")

def plot_mach(t, step):
    """Plot Mach number"""
    ensure_output_dir()

    M = state.state.get_mach_number()

    fig, ax = plt.subplots(figsize=(12, 3))
    cf = ax.contourf(mesh.mesh.X, mesh.mesh.Y, M, levels=40, cmap='jet')
    plt.colorbar(cf, ax=ax, label='Mach Number')
    ax.contour(mesh.mesh.X, mesh.mesh.Y, M, levels=[1.0], colors='white',
               linewidths=1.5, linestyles='--')
    ax.axvline(x=mesh.mesh.Lx/2, color='red', linestyle='--', alpha=0.5)
    ax.set_title(f"Mach Number (FDM) - t={t:.4f}")
    ax.set_xlabel('x')
    ax.set_ylabel('y')

    filename = f"{params.output_dir}/plots/mach_t{t:.4f}.png"
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Mach plot saved: {filename}")

def plot_contours(t, step):
    """Plot 2D contours"""
    ensure_output_dir()

    rho = state.state.rho
    u = state.state.u
    v = state.state.v
    p = state.state.p
    M = state.state.get_mach_number()
    T = p / (rho * params.R)

    fig, axes = plt.subplots(2, 3, figsize=(18, 8))
    fig.suptitle(f"Flow Fields (FDM) - t={t:.4f}", fontsize=14)

    fields = [
        (axes[0,0], rho, 'Density', 'viridis'),
        (axes[0,1], u, 'Velocity u', 'RdBu_r'),
        (axes[0,2], v, 'Velocity v', 'RdBu_r'),
        (axes[1,0], p, 'Pressure', 'plasma'),
        (axes[1,1], T, 'Temperature', 'hot'),
        (axes[1,2], M, 'Mach Number', 'jet')
    ]

    for ax, field, title, cmap in fields:
        cf = ax.contourf(mesh.mesh.X, mesh.mesh.Y, field, levels=40, cmap=cmap)
        plt.colorbar(cf, ax=ax)
        ax.axvline(x=mesh.mesh.Lx/2, color='red', linestyle='--', alpha=0.5)
        ax.set_title(title)
        ax.set_xlabel('x')
        ax.set_ylabel('y')

    plt.tight_layout()
    filename = f"{params.output_dir}/plots/contours_t{t:.4f}.png"
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Contours plot saved: {filename}")

Overwriting plotting.py


## Main solver

In [103]:
%%writefile main.py
"""
Main solver for Sod shock tube - FDM version.
"""

import numpy as np
import time
import os
from parameters import params
import mesh
import state
from initial_conditions import init_sod
from timestep import compute_dt
from rk3 import rk3_step
from plotting import plot_verification, plot_schlieren, plot_mach, plot_contours

def run_simulation():
    """Run Sod shock tube simulation with FDM"""
    print("\n" + "="*60)
    print("SOD SHOCK TUBE SIMULATION - FDM WITH WENO5")
    print("="*60)
    print(f"Grid: {mesh.mesh.Nx} x {mesh.mesh.Ny} nodes")
    print(f"Domain: x∈[0,{mesh.mesh.Lx}], y∈[0,{mesh.mesh.Ly}]")
    print(f"Final time: {params.t_final}")
    print("="*60)

    # Initialize
    init_sod()
    os.makedirs(f"{params.output_dir}/plots", exist_ok=True)

    # Plot initial condition
    print("\n" + "="*60)
    print("PLOTTING INITIAL CONDITION (t=0)")
    print("="*60)
    plot_contours(0.0, 0)
    plot_mach(0.0, 0)
    plot_schlieren(0.0, 0)
    plot_verification(0.0, 0)

    # Simulation loop
    t = 0.0
    step = 0
    t_next_save = params.t_print

    print(f"\n{'Step':>6}  {'t':>9}  {'dt':>10}  {'rho_mean':>10}  {'p_mean':>10}")
    print("-" * 55)

    while t < params.t_final - 1e-10:
        dt = compute_dt(state.state.Q, mesh.mesh.dx, mesh.mesh.dy)
        dt = min(dt, params.t_final - t)

        state.state.Q = rk3_step(state.state.Q, dt, mesh.mesh.dx, mesh.mesh.dy)
        t += dt
        step += 1
        state.state.conservative_to_primitive()

        if step % 10 == 0:
            print(f"{step:>6}  {t:>9.5f}  {dt:>10.3e}  "
                  f"{state.state.rho.mean():>10.4f}  {state.state.p.mean():>10.4f}")

        if t >= t_next_save or abs(t - params.t_final) < 1e-10:
            print(f"\n--- Saving at t={t:.4f}, step={step} ---")
            plot_contours(t, step)
            plot_mach(t, step)
            plot_schlieren(t, step)
            plot_verification(t, step)
            t_next_save += params.t_print
            print("--- Done ---\n")

        if np.any(~np.isfinite(state.state.Q)):
            print(f"NaN detected at step {step}, t={t:.6f}")
            break

    # Plot final condition
    print("\n" + "="*60)
    print("PLOTTING FINAL CONDITION")
    print("="*60)
    plot_contours(t, step)
    plot_mach(t, step)
    plot_schlieren(t, step)
    plot_verification(t, step)
    print("="*60)

    print(f"\nSimulation completed: {step} steps, final time t={t:.5f}")
    return state.state.Q

if __name__ == "__main__":
    start_time = time.time()
    Q_final = run_simulation()
    elapsed = time.time() - start_time
    print(f"\nTotal simulation time: {elapsed:.2f} seconds")
    print(f"Plots saved in: {params.output_dir}/plots/")

Overwriting main.py


## Run the FDM

In [107]:
# Clear old output and run FDM version
!rm -rf output/
!python3 main.py

FDM Grid: 200 x 40 nodes
Domain: x∈[0,10.0], y∈[0,1.0]
dx=0.050251, dy=0.025641

SOD SHOCK TUBE SIMULATION - FDM WITH WENO5
Grid: 200 x 40 nodes
Domain: x∈[0,10.0], y∈[0,1.0]
Final time: 0.5
Sod shock tube initialized (FDM)
  Diaphragm at x = 5.00
  Left:  rho=1.0, p=1.0
  Right: rho=0.125, p=0.1

PLOTTING INITIAL CONDITION (t=0)
Contours plot saved: output/plots/contours_t0.0000.png
Mach plot saved: output/plots/mach_t0.0000.png
Schlieren plot saved: output/plots/schlieren_t0.0000.png
Verification plot saved: output/plots/verification_t0.0000.png

  Step          t          dt    rho_mean      p_mean
-------------------------------------------------------
    10    0.02123   1.701e-03      0.5629      0.5504
    20    0.03862   1.894e-03      0.5630      0.5505

--- Saving at t=0.0511, step=28 ---
Contours plot saved: output/plots/contours_t0.0511.png
Mach plot saved: output/plots/mach_t0.0511.png
Schlieren plot saved: output/plots/schlieren_t0.0511.png
Verification plot saved: output